# Gold Layer: AI Persona Narratives

Generates AI-written investment narratives for every neighbourhood × city × persona
combination using Snowflake Cortex (`mistral-large`).

Reads from `GOLD.BOROUGH_SUMMARY`, `GOLD.INVESTMENT_SCORES`, and `GOLD.REVIEW_THEMES`.
Writes one row per neighbourhood × city × persona to `GOLD.AI_OUTPUTS`.

**Depends on:** Run `gold_borough_summary.ipynb`, `gold_investment_scores.ipynb`,
and `gold_review_themes.ipynb` first.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.AI_OUTPUTS`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
CITIES         = ['BRISTOL', 'LONDON', 'GREATER_MANCHESTER']
DATABASE       = 'AIRBNB_INVESTMENT_DB'
GOLD_SCHEMA    = 'GOLD'
OUTPUT_TABLE   = 'AI_OUTPUTS'
CORTEX_MODEL   = 'mistral-large'
PROMPT_VERSION = 'v1'

PERSONAS = {
    'YIELD_MAXIMISER': {
        'label':       'Yield Maximiser',
        'focus':       'maximising annual revenue, high nightly price, '
                       'strong booking demand',
        'score_col':   'score_yield_maximiser',
        'top_metrics': ['avg_revenue', 'avg_price',
                        'avg_occupancy', 'avg_review_rating'],
    },
    'OCCUPANCY_OPTIMISER': {
        'label':       'Occupancy Optimiser',
        'focus':       'consistent high occupancy, minimal vacancy, '
                       'steady booking flow',
        'score_col':   'score_occupancy_optimiser',
        'top_metrics': ['avg_occupancy', 'avg_revenue',
                        'avg_review_rating', 'pct_superhost'],
    },
    'QUALITY_HOST': {
        'label':       'Quality Host',
        'focus':       'exceptional guest experience, high ratings, '
                       'superhost status, premium positioning',
        'score_col':   'score_quality_host',
        'top_metrics': ['avg_review_rating', 'pct_superhost',
                        'avg_occupancy', 'avg_price'],
    },
}

OUTPUT_TYPES = ['AREA_OVERVIEW', 'LISTING_CANDIDATES', 'RECOMMENDATION']

In [ ]:
def load_gold_tables(session, city):
    df_borough = session.sql(f'''
        SELECT * FROM {DATABASE}.{GOLD_SCHEMA}.BOROUGH_SUMMARY
        WHERE city = '{city}'
    ''').to_pandas()

    df_scores = session.sql(f'''
        SELECT neighbourhood_cleansed,
               AVG(investment_score)  AS avg_investment_score,
               COUNT(*)               AS listing_count,
               AVG(norm_price)        AS avg_norm_price,
               AVG(norm_occupancy)    AS avg_norm_occupancy,
               AVG(norm_revenue)      AS avg_norm_revenue,
               AVG(norm_rating)       AS avg_norm_rating
        FROM {DATABASE}.{GOLD_SCHEMA}.INVESTMENT_SCORES
        WHERE city = '{city}'
        GROUP BY neighbourhood_cleansed
    ''').to_pandas()

    df_themes = session.sql(f'''
        SELECT neighbourhood_cleansed, top_theme,
               pct_mentions_price, pct_mentions_cleanliness,
               pct_mentions_location, pct_mentions_checkin,
               avg_sentiment_score
        FROM {DATABASE}.{GOLD_SCHEMA}.REVIEW_THEMES
        WHERE city = '{city}'
    ''').to_pandas()

    df = df_borough.merge(df_scores, on='neighbourhood_cleansed', how='left')
    df = df.merge(df_themes, on='neighbourhood_cleansed', how='left')

    print(f'Loaded {len(df)} neighbourhoods for {city}')
    return df


def build_metrics_json(row, persona_key):
    persona = PERSONAS[persona_key]
    metrics = {
        'neighbourhood':             row['neighbourhood_cleansed'],
        'persona':                   persona['label'],
        'investment_score':          round(row[persona['score_col']], 2),
        'avg_price':                 round(row['avg_price'], 2),
        'avg_occupancy':             round(row['avg_occupancy'], 4),
        'avg_revenue':               round(row['avg_revenue'], 2),
        'avg_review_rating':         round(row['avg_review_rating'], 2),
        'listing_count':             int(row['listing_count']),
        'pct_superhost':             round(row['pct_superhost'], 2),
        'top_review_theme':          row['top_theme'],
        'pct_mentions_price':        row['pct_mentions_price'],
        'pct_mentions_cleanliness':  row['pct_mentions_cleanliness'],
        'pct_mentions_location':     row['pct_mentions_location'],
        'pct_mentions_checkin':      row['pct_mentions_checkin'],
    }
    return metrics


def _build_user_prompt(metrics, persona_key):
    persona      = PERSONAS[persona_key]
    occ_pct      = round(metrics['avg_occupancy'] * 100, 1)
    rev_fmt      = f"{metrics['avg_revenue']:,.0f}"
    superhost    = round(metrics['pct_superhost'], 1)
    price_pct    = metrics['pct_mentions_price']
    location_pct = metrics['pct_mentions_location']

    return f'''
Neighbourhood: {metrics['neighbourhood']}
Investment Score ({persona['label']}): {metrics['investment_score']}/100
Average Nightly Price: £{metrics['avg_price']}
Estimated Occupancy Rate: {occ_pct}%
Estimated Annual Revenue: £{rev_fmt}
Average Review Rating: {metrics['avg_review_rating']}/5
Total Listings: {metrics['listing_count']}
Superhost Percentage: {superhost}%
Top Review Theme: {metrics['top_review_theme']}
Price Mentions in Reviews: {price_pct:.1f}%
Location Mentions in Reviews: {location_pct:.1f}%
'''


def build_area_overview_prompt(metrics, persona_key):
    persona = PERSONAS[persona_key]

    system_prompt = f"""You are an expert Airbnb investment analyst
advising a {persona['label']} who prioritises {persona['focus']}.

Respond with ONLY a valid JSON object, no other text:
{{
  "investment_summary": "2-3 sentence overall area assessment referencing actual numbers",
  "key_strengths": ["specific strength 1", "specific strength 2", "specific strength 3"],
  "key_risks": ["specific risk 1", "specific risk 2"],
  "confidence": "high if listing_count > 50 and review rating exists, medium if 10-50 listings, low if under 10",
  "recommended_action": "one sentence recommendation for this persona"
}}
Do not mention other personas. Reference actual numbers."""

    user_prompt = _build_user_prompt(metrics, persona_key)
    return system_prompt, user_prompt


def build_listing_candidates_prompt(metrics, persona_key):
    persona = PERSONAS[persona_key]

    system_prompt = f"""You are an expert Airbnb investment analyst
advising a {persona['label']} who prioritises {persona['focus']}.

Based on the neighbourhood data, advise what types of listings to look for as investment candidates.

Respond with ONLY a valid JSON object, no other text:
{{
  "listing_summary": "2 sentence summary of what makes listings in this area promising for this persona",
  "best_property_type": "the most suitable room type for this persona in this area",
  "best_price_range": "recommended nightly price range as a string e.g. £80-£120",
  "what_to_look_for": ["criteria 1", "criteria 2", "criteria 3"],
  "what_to_avoid": ["red flag 1", "red flag 2"],
  "confidence": "high/medium/low"
}}
Reference actual numbers from the data."""

    user_prompt = _build_user_prompt(metrics, persona_key)
    return system_prompt, user_prompt


def build_recommendation_prompt(metrics, persona_key):
    persona = PERSONAS[persona_key]

    system_prompt = f"""You are an expert Airbnb investment analyst
advising a {persona['label']} who prioritises {persona['focus']}.

Generate a short investment recommendation for this neighbourhood.

Respond with ONLY a valid JSON object, no other text:
{{
  "recommendation": "one punchy sentence on why this neighbourhood ranks well for this persona",
  "headline_metric": "the single most compelling number e.g. £18,500 avg annual revenue or 74% occupancy rate",
  "investor_fit": "Strong fit / Moderate fit / Weak fit for {persona['label']}",
  "confidence": "high/medium/low"
}}
Be specific. Reference actual numbers."""

    user_prompt = _build_user_prompt(metrics, persona_key)
    return system_prompt, user_prompt


def call_cortex(session, system_prompt, user_prompt):
    system_escaped = system_prompt.replace("'", "''")
    user_escaped   = user_prompt.replace("'", "''")

    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            '{CORTEX_MODEL}',
            [
                {{'role': 'system', 'content': '{system_escaped}'}},
                {{'role': 'user',   'content': '{user_escaped}'}}
            ]
        ) AS response
    """).to_pandas()

    return result['RESPONSE'][0]


def generate_all_outputs(session, df_merged, city):
    import json

    rows = []
    prompt_builders = {
        'AREA_OVERVIEW':      build_area_overview_prompt,
        'LISTING_CANDIDATES': build_listing_candidates_prompt,
        'RECOMMENDATION':     build_recommendation_prompt,
    }

    for _, row in df_merged.iterrows():
        neighbourhood = row['neighbourhood_cleansed']
        for persona_key in PERSONAS:
            score_col = PERSONAS[persona_key]['score_col']
            metrics   = build_metrics_json(row, persona_key)

            for output_type, builder in prompt_builders.items():
                system_prompt, user_prompt = builder(metrics, persona_key)

                try:
                    ai_response = call_cortex(session, system_prompt, user_prompt)
                    parsed      = json.loads(ai_response)
                    confidence  = parsed.get('confidence', 'medium')
                except json.JSONDecodeError:
                    parsed     = None
                    confidence = 'low'
                    print(f'  WARNING: JSON parse failed — '
                          f'{neighbourhood} / {persona_key} / {output_type}')
                except Exception as e:
                    ai_response = None
                    parsed      = None
                    confidence  = 'error'
                    print(f'  WARNING: Cortex failed — '
                          f'{neighbourhood} / {persona_key} / {output_type}: {e}')

                rows.append({
                    'city':                   city,
                    'neighbourhood_cleansed': neighbourhood,
                    'persona':                persona_key,
                    'output_type':            output_type,
                    'investment_score':       round(row[score_col], 2),
                    'ai_narrative':           ai_response,
                    'confidence':             confidence,
                    'metrics_json':           str(metrics),
                    'prompt_version':         PROMPT_VERSION,
                    'model_used':             CORTEX_MODEL,
                    'computed_at':            pd.Timestamp.now(),
                })
                print(f'  {neighbourhood} / {persona_key} / {output_type} — done')

    return pd.DataFrame(rows)


def write_gold(session, df):
    session.write_pandas(
        df,
        OUTPUT_TABLE,
        database=DATABASE,
        schema=GOLD_SCHEMA,
        overwrite=True,
        auto_create_table=True
    )
    print(f'Written {len(df)} rows to {GOLD_SCHEMA}.{OUTPUT_TABLE}')


def validate(session):
    df_val = session.sql(f"""
        SELECT city, persona, output_type,
               COUNT(*) AS neighbourhood_count,
               COUNT(ai_narrative) AS narratives_generated,
               SUM(CASE WHEN confidence = 'high'
                   THEN 1 ELSE 0 END) AS high_confidence,
               SUM(CASE WHEN confidence = 'medium'
                   THEN 1 ELSE 0 END) AS medium_confidence,
               SUM(CASE WHEN confidence = 'low'
                   THEN 1 ELSE 0 END) AS low_confidence
        FROM {DATABASE}.{GOLD_SCHEMA}.{OUTPUT_TABLE}
        GROUP BY city, persona, output_type
        ORDER BY city, persona, output_type
    """).to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
all_dfs = []
for city in CITIES:
    print(f'\nProcessing {city}...')
    df_merged = load_gold_tables(session, city)
    print(f'  {len(df_merged)} neighbourhoods loaded')
    df_outputs = generate_all_outputs(session, df_merged, city)
    all_dfs.append(df_outputs)
    print(f'  {city} complete \u2014 {len(df_outputs)} rows generated')

df_final = pd.concat(all_dfs, ignore_index=True)
write_gold(session, df_final)
validate(session)

In [ ]:
df_val = session.sql(f"""
    SELECT city, persona, output_type,
           COUNT(*) AS neighbourhood_count,
           COUNT(ai_narrative) AS narratives_generated,
           SUM(CASE WHEN confidence = 'high'
               THEN 1 ELSE 0 END) AS high_confidence,
           SUM(CASE WHEN confidence = 'medium'
               THEN 1 ELSE 0 END) AS medium_confidence,
           SUM(CASE WHEN confidence = 'low'
               THEN 1 ELSE 0 END) AS low_confidence
    FROM {DATABASE}.{GOLD_SCHEMA}.{OUTPUT_TABLE}
    GROUP BY city, persona, output_type
    ORDER BY city, persona, output_type
""").to_pandas()
print(df_val)

## AI Persona Narratives Complete

`GOLD.AI_OUTPUTS` is ready for Streamlit consumption.
Each row represents one neighbourhood × city × persona with a Cortex-generated investment narrative.
If `narratives_generated` is less than `neighbourhood_count` for any group, check the warnings
printed during `generate_all_outputs` — those rows will have `ai_narrative = NULL`.